# 02 — Data Exploration

**Inputs needed:** `data/labels.csv` plus the artefacts from notebook 01.
**Outputs produced:** Inline plots only (no files written).
**Runtime:** Seconds for class balance + statistics; longer if you scroll through slices.


Quick EDA over the prepared dataset:

- Class balance from `data/labels.csv`.
- Per-patient cropped-volume shape & voxel statistics.
- Z-score stats from `crop_metadata.json` (sanity check for normalization).
- Slice viewer with mask overlay (axial).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json
import nibabel as nib
import numpy as np
import pandas as pd

from src.utils.config import load_config
from src.utils.visualization import plot_slice_overlay

cfg = load_config(ROOT / "configs" / "default.yaml")
processed_dir = Path(cfg["paths"]["processed_dir"])
labels = pd.read_csv(cfg["paths"]["labels_csv"])
labels.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ax = sns.countplot(data=labels, x="label")
ax.set_title("Class balance")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Stable cirrhosis (0)", "Future HCC (1)"])
plt.show()

In [ ]:
rows = []
for _, row in labels.iterrows():
    pid = str(row["patient_id"])
    pdir = processed_dir / pid
    vol_path = pdir / "before_cropped.nii.gz"
    meta_path = pdir / "crop_metadata.json"
    if not vol_path.exists() or not meta_path.exists():
        continue
    vol = nib.load(str(vol_path)).get_fdata()
    meta = json.loads(meta_path.read_text())
    z = meta.get("zscore", {})
    rows.append({
        "patient_id": pid,
        "label": int(row["label"]),
        "shape": vol.shape,
        "voxels": int(np.prod(vol.shape)),
        "vol_mean": float(np.mean(vol)),
        "vol_std": float(np.std(vol)),
        "liver_mean": z.get("mean"),
        "liver_std": z.get("std"),
    })
stats = pd.DataFrame(rows)
stats

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
if not stats.empty:
    sns.boxplot(data=stats, x="label", y="vol_mean", ax=axes[0])
    axes[0].set_title("Cropped voxel mean by class")
    sns.boxplot(data=stats, x="label", y="vol_std", ax=axes[1])
    axes[1].set_title("Cropped voxel std by class")
    plt.tight_layout()
plt.show()

## Slice viewer

Overlay liver mask on the central axial slice for a chosen patient.

In [ ]:
if not stats.empty:
    pid = stats.iloc[0]["patient_id"]
    pdir = processed_dir / pid
    vol = nib.load(str(pdir / "before.nii.gz")).get_fdata()
    mask = nib.load(str(pdir / "before_liver.nii.gz")).get_fdata()
    z = vol.shape[-1] // 2
    plot_slice_overlay(vol, mask, slice_idx=z)